<!-- project-navigation -->
# 15_LightGbm
Role: Model experiments and analysis.

See [notebook guide](README.md) and [model comparison](../reports/model_comparison.md). Historical code and outputs are retained; run with the project root as working directory.


In [5]:
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb

from lightgbm import LGBMClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("LightGBM version :", lgb.__version__)

LightGBM version : 4.7.0


In [6]:
X_train, X_valid, y_train, y_valid = joblib.load(
    "data/processed/xgb_split.pkl"
)

print("========== DIMENSIONS ==========")
print("X_train :", X_train.shape)
print("X_valid :", X_valid.shape)
print("y_train :", y_train.shape)
print("y_valid :", y_valid.shape)

========== DIMENSIONS ==========
X_train : (246008, 240)
X_valid : (61503, 240)
y_train : (246008,)
y_valid : (61503,)


In [7]:
# ============================================================
# OPTIMISATION MÉMOIRE
# ============================================================

X_train_lgb = X_train.copy()
X_valid_lgb = X_valid.copy()

for col in X_train_lgb.columns:

    if X_train_lgb[col].dtype == "float64":
        X_train_lgb[col] = X_train_lgb[col].astype("float32")
        X_valid_lgb[col] = X_valid_lgb[col].astype("float32")

    elif X_train_lgb[col].dtype == "int64":
        X_train_lgb[col] = X_train_lgb[col].astype("int32")
        X_valid_lgb[col] = X_valid_lgb[col].astype("int32")

print(
    "Mémoire X_train :",
    round(
        X_train_lgb.memory_usage(deep=True).sum() / 1024**2,
        2
    ),
    "MB"
)

print(
    "Mémoire X_valid :",
    round(
        X_valid_lgb.memory_usage(deep=True).sum() / 1024**2,
        2
    ),
    "MB"
)

Mémoire X_train : 227.1 MB
Mémoire X_valid : 56.78 MB


In [8]:
import re

# Copie pour éviter de modifier X_train/X_valid originaux
X_train_lgb = X_train.copy()
X_valid_lgb = X_valid.copy()

# Nettoyage des noms de colonnes
def clean_feature_names(df):
    df = df.copy()
    
    new_columns = []
    
    for col in df.columns:
        # Convertir en texte
        col = str(col)
        
        # Remplacer tous les caractères problématiques par _
        col = re.sub(r'[^A-Za-z0-9_]', '_', col)
        
        # Éviter les noms qui commencent par un chiffre
        if col[0].isdigit():
            col = "feature_" + col
        
        new_columns.append(col)
    
    df.columns = new_columns
    
    return df


X_train_lgb = clean_feature_names(X_train_lgb)
X_valid_lgb = clean_feature_names(X_valid_lgb)

print("Nombre de variables :", X_train_lgb.shape[1])

print("\nExemples de noms nettoyés :")
print(X_train_lgb.columns[:20].tolist())

Nombre de variables : 240

Exemples de noms nettoyés :
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL']


In [9]:
print("Nombre de doublons :", 
      X_train_lgb.columns.duplicated().sum())

Nombre de doublons : 0


In [10]:
from lightgbm import LGBMClassifier

model_lgb = LGBMClassifier(
    objective="binary",
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    n_jobs=2,
    verbosity=-1
)

print("========== ENTRAÎNEMENT LIGHTGBM ==========")

model_lgb.fit(
    X_train_lgb,
    y_train
)

print("Entraînement terminé !")

========== ENTRAÎNEMENT LIGHTGBM ==========
Entraînement terminé !


In [11]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

y_proba_lgb = model_lgb.predict_proba(
    X_valid_lgb
)[:, 1]

y_pred_lgb = (
    y_proba_lgb >= 0.50
).astype(int)

print("========== MATRICE DE CONFUSION ==========")
print(confusion_matrix(y_valid, y_pred_lgb))

print("\n========== CLASSIFICATION REPORT ==========")
print(
    classification_report(
        y_valid,
        y_pred_lgb,
        digits=4
    )
)

print(
    "\nROC-AUC LightGBM :",
    round(
        roc_auc_score(y_valid, y_proba_lgb),
        4
    )
)

========== MATRICE DE CONFUSION ==========
[[41168 15370]
 [ 1496  3469]]

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

           0     0.9649    0.7281    0.8300     56538
           1     0.1841    0.6987    0.2915      4965

    accuracy                         0.7258     61503
   macro avg     0.5745    0.7134    0.5607     61503
weighted avg     0.9019    0.7258    0.7865     61503


ROC-AUC LightGBM : 0.7817


In [12]:
baseline_lgb_auc = roc_auc_score(
    y_valid,
    y_proba_lgb
)

print(
    f"ROC-AUC LightGBM baseline : {baseline_lgb_auc:.4f}"
)

ROC-AUC LightGBM baseline : 0.7817


In [13]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score
import numpy as np

# ============================================================
# TUNING LIGHTGBM
# ============================================================

lgb_base = LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    random_state=42,
    n_jobs=2,
    verbosity=-1
)

# Espace d'hyperparamètres volontairement limité
param_distributions = {
    "n_estimators": [100, 200],
    "learning_rate": [0.03, 0.05, 0.1],
    "num_leaves": [15, 31, 50],
    "max_depth": [-1, 5, 7],
    "min_child_samples": [20, 50],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_alpha": [0, 0.1],
    "reg_lambda": [1, 2]
}

random_search_lgb = RandomizedSearchCV(
    estimator=lgb_base,
    param_distributions=param_distributions,
    n_iter=8,
    scoring="roc_auc",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1,
    return_train_score=False
)

print("========== DÉBUT DU TUNING LIGHTGBM ==========")

random_search_lgb.fit(
    X_train_lgb,
    y_train
)

print("\n========== TUNING TERMINÉ ==========")

========== DÉBUT DU TUNING LIGHTGBM ==========
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END colsample_bytree=1.0, learning_rate=0.03, max_depth=7, min_child_samples=50, n_estimators=100, num_leaves=50, reg_alpha=0.1, reg_lambda=2, subsample=0.8; total time=  10.9s
[CV] END colsample_bytree=1.0, learning_rate=0.03, max_depth=7, min_child_samples=50, n_estimators=100, num_leaves=50, reg_alpha=0.1, reg_lambda=2, subsample=0.8; total time=   9.6s
[CV] END colsample_bytree=1.0, learning_rate=0.03, max_depth=7, min_child_samples=50, n_estimators=100, num_leaves=50, reg_alpha=0.1, reg_lambda=2, subsample=0.8; total time=   9.2s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=-1, min_child_samples=20, n_estimators=100, num_leaves=50, reg_alpha=0, reg_lambda=2, subsample=1.0; total time=  11.3s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=-1, min_child_samples=20, n_estimators=100, num_leaves=50, reg_alpha=0, reg_lambda=2, subsample=1.0; total 

In [14]:
print("\n========== MEILLEURS HYPERPARAMÈTRES ==========")
print(random_search_lgb.best_params_)

print("\n========== MEILLEUR ROC-AUC CV ==========")
print(
    round(
        random_search_lgb.best_score_,
        4
    )
)


========== MEILLEURS HYPERPARAMÈTRES ==========
{'subsample': 1.0, 'reg_lambda': 2, 'reg_alpha': 0, 'num_leaves': 50, 'n_estimators': 100, 'min_child_samples': 20, 'max_depth': -1, 'learning_rate': 0.1, 'colsample_bytree': 1.0}

========== MEILLEUR ROC-AUC CV ==========
0.7741


In [15]:
best_params_lgb = random_search_lgb.best_params_

print("========== MEILLEURS PARAMÈTRES ==========")
print(best_params_lgb)

model_lgb_opt = LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    random_state=42,
    n_jobs=2,
    verbosity=-1,
    **best_params_lgb
)

print("\n========== ENTRAÎNEMENT LIGHTGBM OPTIMISÉ ==========")

model_lgb_opt.fit(
    X_train_lgb,
    y_train
)

print("Entraînement terminé !")

========== MEILLEURS PARAMÈTRES ==========
{'subsample': 1.0, 'reg_lambda': 2, 'reg_alpha': 0, 'num_leaves': 50, 'n_estimators': 100, 'min_child_samples': 20, 'max_depth': -1, 'learning_rate': 0.1, 'colsample_bytree': 1.0}

========== ENTRAÎNEMENT LIGHTGBM OPTIMISÉ ==========
Entraînement terminé !


In [16]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Probabilités
y_proba_lgb_opt = model_lgb_opt.predict_proba(
    X_valid_lgb
)[:, 1]

# Seuil 0.50
y_pred_lgb_opt = (
    y_proba_lgb_opt >= 0.50
).astype(int)

print("========== MATRICE DE CONFUSION ==========")
print(
    confusion_matrix(
        y_valid,
        y_pred_lgb_opt
    )
)

print("\n========== CLASSIFICATION REPORT ==========")
print(
    classification_report(
        y_valid,
        y_pred_lgb_opt,
        digits=4
    )
)

roc_auc_lgb_opt = roc_auc_score(
    y_valid,
    y_proba_lgb_opt
)

print(
    "\nROC-AUC LightGBM optimisé :",
    round(roc_auc_lgb_opt, 4)
)

========== MATRICE DE CONFUSION ==========
[[41786 14752]
 [ 1572  3393]]

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

           0     0.9637    0.7391    0.8366     56538
           1     0.1870    0.6834    0.2936      4965

    accuracy                         0.7346     61503
   macro avg     0.5754    0.7112    0.5651     61503
weighted avg     0.9010    0.7346    0.7928     61503


ROC-AUC LightGBM optimisé : 0.7819


In [17]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

thresholds = np.arange(
    0.20,
    0.71,
    0.01
)

results_lgb = []

for threshold in thresholds:
    
    y_pred_threshold = (
        y_proba_lgb_opt >= threshold
    ).astype(int)
    
    results_lgb.append({
        "threshold": threshold,
        "accuracy": accuracy_score(
            y_valid,
            y_pred_threshold
        ),
        "precision": precision_score(
            y_valid,
            y_pred_threshold,
            zero_division=0
        ),
        "recall": recall_score(
            y_valid,
            y_pred_threshold,
            zero_division=0
        ),
        "f1": f1_score(
            y_valid,
            y_pred_threshold,
            zero_division=0
        )
    })

threshold_results_lgb = pd.DataFrame(results_lgb)

threshold_results_lgb = threshold_results_lgb.sort_values(
    by="f1",
    ascending=False
)

print(threshold_results_lgb.head(10).to_string(index=False))

 threshold  accuracy  precision   recall       f1
      0.66  0.853292   0.263740 0.456193 0.334243
      0.67  0.858641   0.269616 0.439476 0.334201
      0.68  0.864169   0.275771 0.419738 0.332854
      0.65  0.847227   0.256939 0.471702 0.332670
      0.69  0.869470   0.283166 0.402820 0.332557
      0.64  0.841048   0.251215 0.489225 0.331967
      0.70  0.874803   0.291317 0.384491 0.331481
      0.63  0.834382   0.245193 0.505942 0.330309
      0.62  0.828220   0.240212 0.521450 0.328908
      0.61  0.820871   0.234095 0.536556 0.325971


In [18]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ============================================================
# SEUIL FINAL LIGHTGBM
# ============================================================

best_threshold_lgb = 0.66

y_pred_lgb_final = (
    y_proba_lgb_opt >= best_threshold_lgb
).astype(int)

# ============================================================
# MATRICE DE CONFUSION
# ============================================================

print("========== SEUIL FINAL ==========")
print("Threshold :", best_threshold_lgb)

print("\n========== MATRICE DE CONFUSION ==========")
print(
    confusion_matrix(
        y_valid,
        y_pred_lgb_final
    )
)

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n========== CLASSIFICATION REPORT ==========")

print(
    classification_report(
        y_valid,
        y_pred_lgb_final,
        digits=4
    )
)

# ============================================================
# MÉTRIQUES PRINCIPALES
# ============================================================

print("\n========== MÉTRIQUES FINALES ==========")

print(
    "Accuracy  :",
    round(
        accuracy_score(
            y_valid,
            y_pred_lgb_final
        ),
        4
    )
)

print(
    "Precision :",
    round(
        precision_score(
            y_valid,
            y_pred_lgb_final
        ),
        4
    )
)

print(
    "Recall    :",
    round(
        recall_score(
            y_valid,
            y_pred_lgb_final
        ),
        4
    )
)

print(
    "F1-score  :",
    round(
        f1_score(
            y_valid,
            y_pred_lgb_final
        ),
        4
    )
)

print(
    "ROC-AUC   :",
    round(
        roc_auc_score(
            y_valid,
            y_proba_lgb_opt
        ),
        4
    )
)

========== SEUIL FINAL ==========
Threshold : 0.66

========== MATRICE DE CONFUSION ==========
[[50215  6323]
 [ 2700  2265]]

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

           0     0.9490    0.8882    0.9176     56538
           1     0.2637    0.4562    0.3342      4965

    accuracy                         0.8533     61503
   macro avg     0.6064    0.6722    0.6259     61503
weighted avg     0.8937    0.8533    0.8705     61503


========== MÉTRIQUES FINALES ==========
Accuracy  : 0.8533
Precision : 0.2637
Recall    : 0.4562
F1-score  : 0.3342
ROC-AUC   : 0.7819


In [19]:
# ============================================================
# CALCUL DU SCALE_POS_WEIGHT
# ============================================================

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

print("Nombre de non-défauts :", neg)
print("Nombre de défauts     :", pos)
print("Scale_pos_weight      :", scale_pos_weight)

Nombre de non-défauts : 226148
Nombre de défauts     : 19860
Scale_pos_weight      : 11.38710976837865


In [20]:
# ============================================================
# NETTOYAGE DES NOMS DE VARIABLES POUR LIGHTGBM
# ============================================================

import re

X_train_lgb = X_train.copy()
X_valid_lgb = X_valid.copy()

# Remplacer tous les caractères spéciaux par "_"
X_train_lgb.columns = [
    re.sub(r'[^A-Za-z0-9_]+', '_', str(col))
    for col in X_train_lgb.columns
]

X_valid_lgb.columns = X_train_lgb.columns

print("Nombre de variables :", X_train_lgb.shape[1])
print("Noms des variables nettoyés avec succès.")

Nombre de variables : 240
Noms des variables nettoyés avec succès.


In [21]:
# ============================================================
# LIGHTGBM AVEC SCALE_POS_WEIGHT
# ============================================================

model_lgb_spw = LGBMClassifier(
    objective="binary",

    n_estimators=300,
    learning_rate=0.03,

    num_leaves=50,
    max_depth=-1,
    min_child_samples=20,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=2,

    scale_pos_weight=scale_pos_weight,

    random_state=42,
    n_jobs=2,
    verbosity=-1
)

model_lgb_spw.fit(
    X_train_lgb,
    y_train
)

print("Entraînement terminé.")

Entraînement terminé.


In [22]:
# ============================================================
# PRÉDICTION
# ============================================================

y_proba_lgb_spw = model_lgb_spw.predict_proba(
    X_valid_lgb
)[:, 1]

roc_auc_lgb_spw = roc_auc_score(
    y_valid,
    y_proba_lgb_spw
)

print(
    "ROC-AUC LightGBM avec scale_pos_weight :",
    round(roc_auc_lgb_spw, 4)
)

ROC-AUC LightGBM avec scale_pos_weight : 0.7833


In [23]:
y_pred_lgb_spw = (
    y_proba_lgb_spw >= 0.50
).astype(int)

print("========== MATRICE DE CONFUSION ==========")
print(confusion_matrix(y_valid, y_pred_lgb_spw))

print("\n========== CLASSIFICATION REPORT ==========")
print(
    classification_report(
        y_valid,
        y_pred_lgb_spw,
        digits=4
    )
)

========== MATRICE DE CONFUSION ==========
[[41353 15185]
 [ 1500  3465]]

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

           0     0.9650    0.7314    0.8321     56538
           1     0.1858    0.6979    0.2935      4965

    accuracy                         0.7287     61503
   macro avg     0.5754    0.7147    0.5628     61503
weighted avg     0.9021    0.7287    0.7886     61503



In [24]:
print(
    "ROC-AUC LightGBM avec scale_pos_weight :",
    round(
        roc_auc_score(y_valid, y_proba_lgb_spw),
        4
    )
)

ROC-AUC LightGBM avec scale_pos_weight : 0.7833


In [25]:
y_pred_lgb_spw = (y_proba_lgb_spw >= 0.50).astype(int)

print("========== MATRICE DE CONFUSION ==========")
print(confusion_matrix(y_valid, y_pred_lgb_spw))

print("\n========== CLASSIFICATION REPORT ==========")
print(
    classification_report(
        y_valid,
        y_pred_lgb_spw,
        digits=4
    )
)

print("\n========== ROC-AUC ==========")
print(
    round(
        roc_auc_score(y_valid, y_proba_lgb_spw),
        4
    )
)

========== MATRICE DE CONFUSION ==========
[[41353 15185]
 [ 1500  3465]]

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

           0     0.9650    0.7314    0.8321     56538
           1     0.1858    0.6979    0.2935      4965

    accuracy                         0.7287     61503
   macro avg     0.5754    0.7147    0.5628     61503
weighted avg     0.9021    0.7287    0.7886     61503


========== ROC-AUC ==========
0.7833


In [26]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

thresholds = np.arange(0.50, 0.76, 0.01)

results_threshold_spw = []

for threshold in thresholds:

    y_pred = (y_proba_lgb_spw >= threshold).astype(int)

    results_threshold_spw.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy_score(y_valid, y_pred),
        "precision": precision_score(y_valid, y_pred, zero_division=0),
        "recall": recall_score(y_valid, y_pred, zero_division=0),
        "f1": f1_score(y_valid, y_pred, zero_division=0)
    })

threshold_results_spw = pd.DataFrame(results_threshold_spw)

threshold_results_spw = threshold_results_spw.sort_values(
    by="f1",
    ascending=False
)

print(threshold_results_spw.to_string(index=False))

 threshold  accuracy  precision   recall       f1
      0.68  0.865746   0.279001 0.418530 0.334810
      0.64  0.840918   0.251981 0.493051 0.333515
      0.67  0.859405   0.269875 0.434844 0.333051
      0.66  0.853714   0.263491 0.452367 0.333012
      0.69  0.871177   0.285900 0.397784 0.332688
      0.65  0.847341   0.256976 0.471098 0.332551
      0.63  0.834122   0.245801 0.509970 0.331718
      0.70  0.876591   0.293988 0.377241 0.330452
      0.62  0.827439   0.240011 0.525076 0.329437
      0.71  0.881908   0.303724 0.358107 0.328681
      0.61  0.820562   0.234543 0.540181 0.327073
      0.60  0.812985   0.229003 0.556294 0.324445
      0.72  0.886770   0.312933 0.336757 0.324408
      0.59  0.805196   0.223693 0.572004 0.321613
      0.73  0.891306   0.323770 0.318228 0.320975
      0.58  0.797148   0.218542 0.587311 0.318549
      0.57  0.789457   0.213671 0.600000 0.315121
      0.74  0.895875   0.335918 0.296677 0.315080
      0.56  0.781360   0.208962 0.613293 0.311716


In [27]:
# ============================================================
# SEUIL FINAL LIGHTGBM + SCALE_POS_WEIGHT
# ============================================================

threshold_final_spw = 0.68

y_pred_lgb_spw_final = (
    y_proba_lgb_spw >= threshold_final_spw
).astype(int)

print("========== SEUIL FINAL ==========")
print("Threshold :", threshold_final_spw)

print("\n========== MATRICE DE CONFUSION ==========")
print(
    confusion_matrix(
        y_valid,
        y_pred_lgb_spw_final
    )
)

print("\n========== CLASSIFICATION REPORT ==========")
print(
    classification_report(
        y_valid,
        y_pred_lgb_spw_final,
        digits=4
    )
)

print("\n========== MÉTRIQUES FINALES ==========")

print(
    "Accuracy  :",
    round(
        accuracy_score(y_valid, y_pred_lgb_spw_final),
        4
    )
)

print(
    "Precision :",
    round(
        precision_score(y_valid, y_pred_lgb_spw_final),
        4
    )
)

print(
    "Recall    :",
    round(
        recall_score(y_valid, y_pred_lgb_spw_final),
        4
    )
)

print(
    "F1-score  :",
    round(
        f1_score(y_valid, y_pred_lgb_spw_final),
        4
    )
)

print(
    "ROC-AUC   :",
    round(
        roc_auc_score(y_valid, y_proba_lgb_spw),
        4
    )
)

========== SEUIL FINAL ==========
Threshold : 0.68

========== MATRICE DE CONFUSION ==========
[[51168  5370]
 [ 2887  2078]]

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

           0     0.9466    0.9050    0.9253     56538
           1     0.2790    0.4185    0.3348      4965

    accuracy                         0.8657     61503
   macro avg     0.6128    0.6618    0.6301     61503
weighted avg     0.8927    0.8657    0.8777     61503


========== MÉTRIQUES FINALES ==========
Accuracy  : 0.8657
Precision : 0.279
Recall    : 0.4185
F1-score  : 0.3348
ROC-AUC   : 0.7833


In [28]:
# ============================================================
# SAUVEGARDE DU MEILLEUR MODÈLE LIGHTGBM ACTUEL
# ============================================================

import os
import joblib

os.makedirs("data/models", exist_ok=True)

# Sauvegarde du modèle
joblib.dump(
    model_lgb_spw,
    "data/models/lightgbm_scale_pos_weight.pkl",
    compress=3
)

print("Modèle LightGBM sauvegardé avec succès.")
print("ROC-AUC :", round(roc_auc_score(y_valid, y_proba_lgb_spw), 4))
print("Seuil final : 0.68")

Modèle LightGBM sauvegardé avec succès.
ROC-AUC : 0.7833
Seuil final : 0.68


In [29]:
import os

print("Répertoire de travail actuel :")
print(os.getcwd())

print("\nContenu de data/models :")

if os.path.exists("data/models"):
    print(os.listdir("data/models"))
else:
    print("Le dossier data/models n'existe pas.")

Répertoire de travail actuel :
c:\Users\Asus\OneDrive\Bureau\FinTech

Contenu de data/models :
['lightgbm_feature_dtypes.pkl', 'lightgbm_feature_names.pkl', 'lightgbm_metadata.pkl', 'lightgbm_scale_pos_weight.pkl']


In [30]:
import os

model_path = os.path.abspath(
    "data/models/lightgbm_scale_pos_weight.pkl"
)

print("Chemin complet du modèle :")
print(model_path)

print("\nLe fichier existe :", os.path.exists(model_path))

Chemin complet du modèle :
c:\Users\Asus\OneDrive\Bureau\FinTech\data\models\lightgbm_scale_pos_weight.pkl

Le fichier existe : True


In [31]:
import joblib
import pandas as pd

# ============================================================
# 1. RECHARGEMENT DU MODÈLE LIGHTGBM FINAL
# ============================================================

model_lgb_spw = joblib.load(
    "data/models/lightgbm_scale_pos_weight.pkl"
)

print("Modèle LightGBM chargé avec succès !")
print(type(model_lgb_spw))

Modèle LightGBM chargé avec succès !
<class 'lightgbm.sklearn.LGBMClassifier'>


In [32]:
# ============================================================
# 2. CHARGEMENT DES DONNÉES DE VALIDATION
# ============================================================

X_train, X_valid, y_train, y_valid = joblib.load(
    "data/processed/xgb_split.pkl"
)

print("X_valid :", X_valid.shape)
print("y_valid :", y_valid.shape)

X_valid : (61503, 240)
y_valid : (61503,)


In [33]:
import re

# ============================================================
# 3. NETTOYAGE DES NOMS DE VARIABLES
# ============================================================

X_valid_lgb = X_valid.copy()

X_valid_lgb.columns = [
    re.sub(r'[^A-Za-z0-9_]+', '_', str(col))
    for col in X_valid_lgb.columns
]

print("Nombre de variables :", X_valid_lgb.shape[1])
print("Nombre de doublons :", X_valid_lgb.columns.duplicated().sum())

Nombre de variables : 240
Nombre de doublons : 0


In [34]:
# ============================================================
# 4. ÉVALUATION LIGHTGBM + SCALE_POS_WEIGHT
# SEUIL 0.50
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Probabilités
y_valid_proba = model_lgb_spw.predict_proba(X_valid_lgb)[:, 1]

# Seuil standard
threshold = 0.50

# Prédictions
y_valid_pred = (y_valid_proba >= threshold).astype(int)

print("============================================================")
print("LIGHTGBM + SCALE_POS_WEIGHT — SEUIL 0.50")
print("============================================================")

print(f"Accuracy  : {accuracy_score(y_valid, y_valid_pred):.4f}")
print(f"Precision : {precision_score(y_valid, y_valid_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_valid, y_valid_pred, zero_division=0):.4f}")
print(f"F1-score  : {f1_score(y_valid, y_valid_pred, zero_division=0):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_valid, y_valid_proba):.4f}")

print("\n========== MATRICE DE CONFUSION ==========")
print(confusion_matrix(y_valid, y_valid_pred))

print("\n========== CLASSIFICATION REPORT ==========")
print(classification_report(
    y_valid,
    y_valid_pred,
    digits=4,
    zero_division=0
))

LIGHTGBM + SCALE_POS_WEIGHT — SEUIL 0.50
Accuracy  : 0.7287
Precision : 0.1858
Recall    : 0.6979
F1-score  : 0.2935
ROC-AUC   : 0.7833

========== MATRICE DE CONFUSION ==========
[[41353 15185]
 [ 1500  3465]]

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

           0     0.9650    0.7314    0.8321     56538
           1     0.1858    0.6979    0.2935      4965

    accuracy                         0.7287     61503
   macro avg     0.5754    0.7147    0.5628     61503
weighted avg     0.9021    0.7287    0.7886     61503



In [35]:
# ============================================================
# RECHERCHE DU SEUIL OPTIMAL — LIGHTGBM + SCALE_POS_WEIGHT
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ------------------------------------------------------------
# 1. Tester les seuils
# ------------------------------------------------------------

thresholds = np.arange(0.20, 0.81, 0.01)

threshold_results = []

for threshold in thresholds:

    y_pred = (y_valid_proba >= threshold).astype(int)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy_score(y_valid, y_pred),
        "precision": precision_score(
            y_valid, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_valid, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_valid, y_pred, zero_division=0
        )
    })

threshold_results = pd.DataFrame(threshold_results)

# ------------------------------------------------------------
# 2. Meilleurs seuils selon différentes métriques
# ------------------------------------------------------------

print("============================================================")
print("MEILLEURS SEUILS")
print("============================================================")

best_f1 = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_recall = threshold_results.loc[
    threshold_results["recall"].idxmax()
]

best_precision = threshold_results.loc[
    threshold_results["precision"].idxmax()
]

best_accuracy = threshold_results.loc[
    threshold_results["accuracy"].idxmax()
]

print("\n--- Meilleur F1-score ---")
print(best_f1)

print("\n--- Meilleur Recall ---")
print(best_recall)

print("\n--- Meilleure Precision ---")
print(best_precision)

print("\n--- Meilleure Accuracy ---")
print(best_accuracy)

# ------------------------------------------------------------
# 3. Tableau complet des meilleurs seuils
# ------------------------------------------------------------

print("\n============================================================")
print("TOP 15 SEUILS SELON LE F1-SCORE")
print("============================================================")

print(
    threshold_results
    .sort_values("f1", ascending=False)
    .head(15)
    .to_string(index=False)
)

MEILLEURS SEUILS

--- Meilleur F1-score ---
threshold    0.680000
accuracy     0.865746
precision    0.279001
recall       0.418530
f1           0.334810
Name: 48, dtype: float64

--- Meilleur Recall ---
threshold    0.200000
accuracy     0.327805
precision    0.103019
recall       0.950655
f1           0.185893
Name: 0, dtype: float64

--- Meilleure Precision ---
threshold    0.810000
accuracy     0.916573
precision    0.450829
recall       0.153273
f1           0.228769
Name: 61, dtype: float64

--- Meilleure Accuracy ---
threshold    0.810000
accuracy     0.916573
precision    0.450829
recall       0.153273
f1           0.228769
Name: 61, dtype: float64

TOP 15 SEUILS SELON LE F1-SCORE
 threshold  accuracy  precision   recall       f1
      0.68  0.865746   0.279001 0.418530 0.334810
      0.64  0.840918   0.251981 0.493051 0.333515
      0.67  0.859405   0.269875 0.434844 0.333051
      0.66  0.853714   0.263491 0.452367 0.333012
      0.69  0.871177   0.285900 0.397784 0.332688
  

In [36]:
# ============================================================
# VALIDATION FINALE DU SEUIL 0.68
# LIGHTGBM + SCALE_POS_WEIGHT
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Seuil retenu
final_threshold = 0.68

# Prédictions finales
y_pred_final = (
    y_valid_proba >= final_threshold
).astype(int)

# ------------------------------------------------------------
# MÉTRIQUES
# ------------------------------------------------------------

accuracy_final = accuracy_score(
    y_valid,
    y_pred_final
)

precision_final = precision_score(
    y_valid,
    y_pred_final,
    zero_division=0
)

recall_final = recall_score(
    y_valid,
    y_pred_final,
    zero_division=0
)

f1_final = f1_score(
    y_valid,
    y_pred_final,
    zero_division=0
)

roc_auc_final = roc_auc_score(
    y_valid,
    y_valid_proba
)

# ------------------------------------------------------------
# AFFICHAGE
# ------------------------------------------------------------

print("============================================================")
print("LIGHTGBM FINAL — SEUIL 0.68")
print("============================================================")

print(f"Threshold : {final_threshold:.2f}")
print(f"Accuracy  : {accuracy_final:.4f}")
print(f"Precision : {precision_final:.4f}")
print(f"Recall    : {recall_final:.4f}")
print(f"F1-score  : {f1_final:.4f}")
print(f"ROC-AUC   : {roc_auc_final:.4f}")

# ------------------------------------------------------------
# MATRICE DE CONFUSION
# ------------------------------------------------------------

cm = confusion_matrix(
    y_valid,
    y_pred_final
)

print("\n========== MATRICE DE CONFUSION ==========")
print(cm)

# ------------------------------------------------------------
# CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\n========== CLASSIFICATION REPORT ==========")

print(
    classification_report(
        y_valid,
        y_pred_final,
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# DÉTAIL TN / FP / FN / TP
# ------------------------------------------------------------

tn, fp, fn, tp = cm.ravel()

print("\n========== DÉTAIL DES PRÉDICTIONS ==========")
print(f"TN — Non-défaillants correctement classés : {tn:,}")
print(f"FP — Faux positifs                     : {fp:,}")
print(f"FN — Défauts non détectés              : {fn:,}")
print(f"TP — Défauts correctement détectés     : {tp:,}")

LIGHTGBM FINAL — SEUIL 0.68
Threshold : 0.68
Accuracy  : 0.8657
Precision : 0.2790
Recall    : 0.4185
F1-score  : 0.3348
ROC-AUC   : 0.7833

========== MATRICE DE CONFUSION ==========
[[51168  5370]
 [ 2887  2078]]

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

           0     0.9466    0.9050    0.9253     56538
           1     0.2790    0.4185    0.3348      4965

    accuracy                         0.8657     61503
   macro avg     0.6128    0.6618    0.6301     61503
weighted avg     0.8927    0.8657    0.8777     61503


========== DÉTAIL DES PRÉDICTIONS ==========
TN — Non-défaillants correctement classés : 51,168
FP — Faux positifs                     : 5,370
FN — Défauts non détectés              : 2,887
TP — Défauts correctement détectés     : 2,078


In [37]:
import joblib

model_lgb_spw = joblib.load(
    "data/models/lightgbm_scale_pos_weight.pkl"
)

print("Modèle chargé :", type(model_lgb_spw))

Modèle chargé : <class 'lightgbm.sklearn.LGBMClassifier'>


In [38]:
print("X_train :", X_train.shape)
print("X_valid :", X_valid.shape)
print("y_train :", y_train.shape)
print("y_valid :", y_valid.shape)

X_train : (246008, 240)
X_valid : (61503, 240)
y_train : (246008,)
y_valid : (61503,)


In [39]:
from sklearn.metrics import roc_auc_score

# ============================================================
# OVERFITTING / UNDERFITTING - LIGHTGBM FINAL
# ============================================================

# Probabilités sur TRAIN
y_train_proba = model_lgb_spw.predict_proba(X_train)[:, 1]

# Probabilités sur VALIDATION
y_valid_proba = model_lgb_spw.predict_proba(X_valid)[:, 1]

# ROC-AUC
auc_train = roc_auc_score(y_train, y_train_proba)
auc_valid = roc_auc_score(y_valid, y_valid_proba)

ecart = auc_train - auc_valid

print("========== OVERFITTING / UNDERFITTING ==========")
print(f"ROC-AUC TRAIN      : {auc_train:.4f}")
print(f"ROC-AUC VALIDATION : {auc_valid:.4f}")
print(f"Écart Train-Val    : {ecart:.4f}")

========== OVERFITTING / UNDERFITTING ==========
ROC-AUC TRAIN      : 0.8394
ROC-AUC VALIDATION : 0.7833
Écart Train-Val    : 0.0561


In [40]:
print("Modèle :", type(model_lgb_spw))
print("X_train :", X_train.shape)
print("X_valid :", X_valid.shape)

Modèle : <class 'lightgbm.sklearn.LGBMClassifier'>
X_train : (246008, 240)
X_valid : (61503, 240)


In [41]:
print("========== HYPERPARAMÈTRES LIGHTGBM FINAL ==========")

params_lgb = model_lgb_spw.get_params()

for param, value in params_lgb.items():
    print(f"{param} : {value}")

========== HYPERPARAMÈTRES LIGHTGBM FINAL ==========
boosting_type : gbdt
class_weight : None
colsample_bytree : 0.8
importance_type : split
learning_rate : 0.03
max_depth : -1
min_child_samples : 20
min_child_weight : 0.001
min_split_gain : 0.0
n_estimators : 300
n_jobs : 2
num_leaves : 50
objective : binary
random_state : 42
reg_alpha : 0.1
reg_lambda : 2
subsample : 0.8
subsample_for_bin : 200000
subsample_freq : 0
scale_pos_weight : 11.38710976837865
verbosity : -1
